In [47]:
#Step 1: Data Exploration & Leading
#importing dependencies
import pandas as pd



In [4]:
df = pd.read_csv('Walmart.csv',encoding_errors='ignore')
df.shape

(10051, 20)

In [5]:
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,hour,day_of_week,month,Revenue,Profit,price_range,time_segment,is_weekend,transaction_size
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,2019-05-01,1900-01-01 13:08:00,Ewallet,9.1,0.48,13,Wednesday,May,522.83,250.9584,High,Afternoon,False,Very Large
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,2019-08-03,1900-01-01 10:29:00,Cash,9.6,0.48,10,Saturday,August,76.40,36.6720,Low,Morning,True,Medium
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,2019-03-03,1900-01-01 13:23:00,Credit card,7.4,0.33,13,Sunday,March,324.31,107.0223,Medium,Afternoon,True,Large
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,2019-01-27,1900-01-01 20:33:00,Ewallet,8.4,0.33,20,Sunday,January,465.76,153.7008,High,Evening,True,Large
4,5,WALM013,Irving,Sports and travel,86.31,7.0,2019-08-02,1900-01-01 10:37:00,Ewallet,5.3,0.48,10,Friday,August,604.17,290.0016,High,Morning,False,Very Large


In [6]:
df.describe()

,invoice_id,unit_price,quantity,rating,profit_margin,hour,Revenue,Profit
count,10051.000000,10020.000000,10020.000000,10051.000000,10051.000000,10051.000000,10020.000000,10020.000000
mean,5025.741220,50.630053,2.353493,5.825659,0.393791,15.095712,121.240058,47.715273
std,2901.174372,21.197783,1.602658,1.763991,0.090669,4.029813,112.467617,47.095401
min,1.000000,10.080000,1.000000,3.000000,0.180000,6.000000,10.170000,2.700000
25%,2513.500000,32.000000,1.000000,4.000000,0.330000,12.000000,54.000000,20.460000
50%,5026.000000,51.000000,2.000000,6.000000,0.330000,16.000000,88.000000,34.650000
75%,7538.500000,69.000000,3.000000,7.000000,0.480000,18.000000,156.000000,60.480000
max,10000.000000,99.960000,10.000000,10.000000,0.570000,23.000000,993.000000,507.716100


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   invoice_id        10051 non-null  int64  
 1   Branch            10051 non-null  str    
 2   City              10051 non-null  str    
 3   category          10051 non-null  str    
 4   unit_price        10020 non-null  float64
 5   quantity          10020 non-null  float64
 6   date              10051 non-null  str    
 7   time              10051 non-null  str    
 8   payment_method    10051 non-null  str    
 9   rating            10051 non-null  float64
 10  profit_margin     10051 non-null  float64
 11  hour              10051 non-null  int64  
 12  day_of_week       10051 non-null  str    
 13  month             10051 non-null  str    
 14  Revenue           10020 non-null  float64
 15  Profit            10020 non-null  float64
 16  price_range       10020 non-null  str    
 17  time

In [8]:
df.duplicated().sum()

np.int64(51)

In [9]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [10]:
df.isnull().sum()

invoice_id           0
Branch               0
City                 0
category             0
unit_price          31
quantity            31
date                 0
time                 0
payment_method       0
rating               0
profit_margin        0
hour                 0
day_of_week          0
month                0
Revenue             31
Profit              31
price_range         31
time_segment         0
is_weekend           0
transaction_size    31
dtype: int64

In [11]:
#dropping all rows with missing values
df.dropna(inplace=True)
df.shape

(9969, 20)

In [48]:
import duckdb

query = """
SELECT city,
       SUM(unit_price * quantity) AS revenue
FROM 'Walmart.csv'
GROUP BY city
ORDER BY revenue DESC
"""

result = duckdb.query(query).to_df()

print(result)

            City   revenue
0        Weslaco  46644.79
1     Waxahachie  40966.33
2          Plano  25712.34
3    San Antonio  25001.56
4     Richardson  24690.60
..           ...       ...
93      Longview   6769.33
94      Pearland   6572.91
95        Irving   6237.11
96    Lewisville   5568.84
97  Lake Jackson   5038.90

[98 rows x 2 columns]


In [55]:

#finding different payment methods and no.of transcations with no of qty sold.
duckdb.query("""
 SELECT payment_method,
 COUNT(*) as no_payments,
SUM(quantity) as no_qty_sold
FROM walmart.csv
GROUP BY payment_method""").df()

,payment_method,no_payments,no_qty_sold
0,Credit card,4260,9573.0
1,Ewallet,3911,8932.0
2,Cash,1880,5077.0


In [54]:
duckdb.query("""
  SELECT COUNT(DISTINCT branch)
             FROM walmart.csv """).df()

,count(DISTINCT branch)
0,100


In [53]:
duckdb.query(" SELECT MIN(quantity) FROM walmart.csv").df()

,min(quantity)
0,1.0


In [52]:

#Identify the highest rated category in each branch, displaying the brach, category , avg rating
duckdb.query("""
SELECT *
FROM
(    SELECT
        Branch,
        Category,
        AVG(Rating) AS avg_rating,
        RANK() OVER(PARTITION BY Branch ORDER BY AVG(rating) DESC) as rank
FROM 'Walmart.csv'
GROUP BY Branch, Category
)
             WHERE rank = 1
""").df()

,Branch,category,avg_rating,rank
0,WALM015,Home and lifestyle,6.223077,1
1,WALM063,Electronic accessories,8.325000,1
2,WALM064,Health and beauty,8.150000,1
3,WALM067,Sports and travel,9.700000,1
4,WALM080,Electronic accessories,7.200000,1
...,...,...,...,...
96,WALM062,Sports and travel,7.300000,1
97,WALM014,Electronic accessories,6.833333,1
98,WALM023,Health and beauty,8.000000,1
99,WALM054,Health and beauty,5.600000,1


In [51]:
#identify the busiest day for each branch based on the no.of transctions.
duckdb.query("""
SELECT 
    Branch,
    day_of_week,
    COUNT(*) AS no_transactions
FROM walmart.csv
GROUP BY Branch, day_of_week
ORDER BY no_transactions DESC
LIMIT 10
""").df()

,Branch,day_of_week,no_transactions
0,WALM058,Sunday,48
1,WALM082,Thursday,41
2,WALM030,Wednesday,41
3,WALM084,Sunday,39
4,WALM084,Tuesday,39
5,WALM058,Saturday,39
6,WALM009,Tuesday,39
7,WALM009,Wednesday,39
8,WALM069,Monday,37
9,WALM069,Thursday,37


In [50]:
#Determine the average, minimum , and maximum rating of products for category each city. 
duckdb.query("""
SELECT 
     city,
     category,
     MIN(rating) as min_rating,
     MAX(rating) as max_rating,
     AVG(rating) as avg_rating,
FROM walmart.csv
GROUP BY city, category
""").df()


,City,category,min_rating,max_rating,avg_rating
0,Harlingen,Electronic accessories,9.6,9.6,9.600000
1,Texas City,Food and beverages,5.5,5.9,5.700000
2,Pharr,Electronic accessories,4.8,4.8,4.800000
3,Cleburne,Health and beauty,4.2,5.6,4.875000
4,Garland,Food and beverages,4.2,9.9,6.871429
...,...,...,...,...,...
508,Odessa,Home and lifestyle,3.0,9.0,6.190476
509,Corpus Christi,Fashion accessories,3.0,9.0,6.000000
510,Frisco,Home and lifestyle,3.0,9.0,6.636364
511,Texas City,Electronic accessories,3.0,7.0,4.571429


In [56]:
#calculate the total profit for each category by considering total_profit ass (unit_price * quantity * profit_margin).
# List category and total_profit, ordered  from highest to lowest profit.\
duckdb.query("""
    SELECT 
       category,
       SUM(revenue) as total_revenue,
        SUM(revenue * profit_margin) as profit
    FROM walmart.csv
    GROUP BY category 
""").df()


,category,total_revenue,profit
0,Food and beverages,53471.28,21552.8622
1,Health and beauty,46851.18,18671.7345
2,Sports and travel,52497.93,20613.8082
3,Fashion accessories,491833.90,193244.5332
4,Home and lifestyle,491996.06,193251.6081
5,Electronic accessories,78175.03,30772.4895


In [57]:
# Determine the most common payment method for each branch. Display branch and the preferred payment method.
duckdb.query("""
    SELECT 
        Branch,
        payment_method,
        COUNT() as total_trans,
    FROM walmart.csv
    GROUP BY Branch,payment_method      
    """ ).df()

,Branch,payment_method,total_trans
0,WALM013,Ewallet,44
1,WALM088,Ewallet,55
2,WALM065,Credit card,102
3,WALM035,Cash,65
4,WALM027,Ewallet,50
...,...,...,...
287,WALM099,Ewallet,8
288,WALM092,Credit card,11
289,WALM038,Ewallet,12
290,WALM011,Ewallet,40


In [58]:
duckdb.query("""
   SELECT 
     Branch,
     time_segment,
     COUNT (*)
    FROM walmart.csv
        GROUP BY Branch, time_segment          
""").df()

,Branch,time_segment,count_star()
0,WALM083,Morning,11
1,WALM023,Afternoon,37
2,WALM046,Morning,43
3,WALM012,Evening,26
4,WALM075,Afternoon,108
...,...,...,...
379,WALM053,Night,1
380,WALM094,Night,2
381,WALM003,Night,10
382,WALM074,Night,12


In [59]:
# identify 5 branch with highest decrease ratio in revenue compare to last year( current year 2023 qnd last year 2022)
#rdr = last_rev-cr_rev/ls_rev*100
duckdb.query("""
WITH revenue_by_year AS (
    SELECT 
        Branch,
        EXTRACT(YEAR FROM CAST(date AS DATE)) AS year,
        SUM(Revenue) AS revenue
    FROM walmart.csv
    GROUP BY Branch, year
),
pivot_data AS (
    SELECT
        Branch,
        SUM(CASE WHEN year = 2022 THEN revenue END) AS revenue_2022,
        SUM(CASE WHEN year = 2023 THEN revenue END) AS revenue_2023
    FROM revenue_by_year
    GROUP BY Branch
)
SELECT 
    Branch,
    ROUND(revenue_2022) AS revenue_2022,
    ROUND(revenue_2023) AS revenue_2023,
    ROUND((revenue_2022 - revenue_2023) * 100.0 / revenue_2022, 2) AS decrease_ratio_percent
FROM pivot_data
WHERE revenue_2022 IS NOT NULL 
  AND revenue_2023 IS NOT NULL
ORDER BY decrease_ratio_percent DESC
LIMIT 5
""").df()

,Branch,revenue_2022,revenue_2023,decrease_ratio_percent
0,WALM045,1731.0,647.0,62.62
1,WALM047,2581.0,1069.0,58.58
2,WALM098,2446.0,1030.0,57.89
3,WALM033,2099.0,931.0,55.65
4,WALM081,1723.0,850.0,50.67


In [ ]:
#High Revenue but Low Profit Categories
duckdb.query("""
SELECT 
    category,
    SUM(Revenue) AS total_revenue,
    SUM(Profit) AS total_profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Revenue), 2) AS profit_margin_pct
FROM walmart.csv
GROUP BY category
ORDER BY profit_margin_pct ASC
LIMIT 5
""").df()

,category,total_revenue,total_profit,profit_margin_pct
0,Sports and travel,52497.93,20613.8082,39.27
1,Home and lifestyle,491996.06,193251.6081,39.28
2,Fashion accessories,491833.90,193244.5332,39.29
3,Electronic accessories,78175.03,30772.4895,39.36
4,Health and beauty,46851.18,18671.7345,39.85


In [ ]:
#High Traffic but Low Revenue Branches
duckdb.query("""
SELECT 
    Branch,
    COUNT(*) AS transactions,
    SUM(Revenue) AS revenue,
    ROUND(SUM(Revenue) * 1.0 / COUNT(*), 2) AS revenue_per_transaction
FROM walmart.csv
GROUP BY Branch
ORDER BY revenue_per_transaction ASC
LIMIT 5
""").df()

,Branch,transactions,revenue,revenue_per_transaction
0,WALM092,51,5038.90,98.80
1,WALM031,56,5568.84,99.44
2,WALM048,80,8130.80,101.64
3,WALM058,240,24650.37,102.71
4,WALM036,73,7688.50,105.32


In [ ]:
#High Rating but Low Revenue Categories
duckdb.query("""
SELECT 
    category,
    AVG(rating) AS avg_rating,
    SUM(Revenue) AS total_revenue
FROM walmart.csv
GROUP BY category
ORDER BY avg_rating DESC, total_revenue ASC
LIMIT 5
""").df()

,category,avg_rating,total_revenue
0,Food and beverages,7.113218,53471.28
1,Health and beauty,7.003289,46851.18
2,Sports and travel,6.916265,52497.93
3,Electronic accessories,5.912172,78175.03
4,Fashion accessories,5.778161,491833.90


In [ ]:
#Branches with Revenue Drop (Month-wise)
duckdb.query("""
SELECT *
FROM (
    SELECT 
        Branch,
        month,
        SUM(Revenue) AS revenue,
        LAG(SUM(Revenue)) OVER (
            PARTITION BY Branch 
            ORDER BY month
        ) AS prev_month_revenue,
        ROUND(
            (SUM(Revenue) - LAG(SUM(Revenue)) OVER (
                PARTITION BY Branch 
                ORDER BY month
            )) * 100.0 
            / LAG(SUM(Revenue)) OVER (
                PARTITION BY Branch 
                ORDER BY month
            ), 2
        ) AS pct_change
    FROM walmart.csv
    GROUP BY Branch, month
)
WHERE pct_change < -30
""").df()

,Branch,month,revenue,prev_month_revenue,pct_change
0,WALM034,December,376.00,972.16,-61.32
1,WALM034,July,203.00,974.72,-79.17
2,WALM034,March,242.96,486.00,-50.01
3,WALM034,November,261.00,646.60,-59.64
4,WALM034,September,477.00,1211.83,-60.64
...,...,...,...,...,...
389,WALM054,September,537.00,975.00,-44.92
390,WALM056,February,1207.82,2857.00,-57.72
391,WALM056,June,637.56,1149.39,-44.53
392,WALM056,May,968.18,2358.70,-58.95


In [ ]:
#Unusual High Transactions but Low Profit
duckdb.query("""
SELECT 
    Branch,
    COUNT(*) AS transactions,
    SUM(Profit) AS total_profit,
    ROUND(SUM(Profit) * 1.0 / COUNT(*), 2) AS profit_per_txn
FROM walmart.csv
GROUP BY Branch
ORDER BY transactions DESC, profit_per_txn ASC
LIMIT 5
""").df()

,Branch,transactions,total_profit,profit_per_txn
0,WALM058,240,8134.6221,33.89
1,WALM009,238,12341.9232,51.86
2,WALM030,233,11851.4880,50.86
3,WALM069,224,8007.0210,35.75
4,WALM074,212,8491.6986,40.06


In [ ]:
#High Rating but Low Profit Products
duckdb.query("""
SELECT 
    category,
    AVG(rating) AS avg_rating,
    SUM(Profit) AS total_profit
FROM walmart.csv
GROUP BY category
ORDER BY avg_rating DESC, total_profit ASC
LIMIT 5
""").df()

,category,avg_rating,total_profit
0,Food and beverages,7.113218,21552.8622
1,Health and beauty,7.003289,18671.7345
2,Sports and travel,6.916265,20613.8082
3,Electronic accessories,5.912172,30772.4895
4,Fashion accessories,5.778161,193244.5332


In [ ]:
#Weekend vs Weekday Behavior
duckdb.query("""
SELECT 
    is_weekend,
    COUNT(*) AS transactions,
    SUM(Revenue) AS revenue,
    ROUND(SUM(Revenue) * 1.0 / COUNT(*), 2) AS avg_order_value
FROM walmart.csv
GROUP BY is_weekend
""").df()

,is_weekend,transactions,revenue,avg_order_value
0,True,2908,360556.89,123.99
1,False,7143,854268.49,119.60
